# 08 Nasaan ito? Mga mapa — notebook

Ang unang tanong ng Mayor tungkol sa anumang outbreak ay *"saan?"* Sa notebook na ito, gagawin mo ang mga sumusunod:

1. i-join ang bilang ng dengue sa 24 na barangay polygon ng **Santa Luz City** (kathang-isip; `santa_luz_barangays.geojson`) at kuwentahin ang **attack rate**;
2. imapa nang magkatabi ang bilang at ang rate at alamin kung bakit magkaiba ang kuwento nila;
3. pumili nang sadya ng **class break** at **color palette**;
4. makita kung gaano kagalaw ang rate kapag maliliit ang numero;
5. bumuo ng interactive na mapa gamit ang plotly;
6. hulihin ang isang join na tahimik na nawalan ng dalawang barangay;
7. mag-zoom out sa totoong **18 rehiyon ng Pilipinas** para sa konteksto;
8. maghanda ng mapang ligtas i-publish (small-cell suppression).

Gawa-gawa ang lahat tungkol sa Santa Luz; totoo at may source ang mapa ng mga rehiyon at ang statistics nito.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## Hakbang 1 — Bilang at rate bawat barangay

Dalawang bagay ang kailangan natin bawat barangay: ang bilang ng kaso ng dengue (mula sa malinis na line list ng [kabanata 03](../03_clean_linelist.md)) at ang populasyon (mula sa file ng mga barangay). Tapos, **attack rate bawat 10,000 = kaso ÷ populasyon × 10,000**.

> **Tanungin si Tala:** *May GeoDataFrame akong `brgy` (24 na row; mga column na `barangay_id`, `barangay`, `band`, `population`, `geometry`) at isang line list na `ll` (isang row bawat kaso; mga column na `barangay_id`, `disease`). Gawa-gawang halimbawa: mga row ng `ll` na `("SL01", "dengue")`, `("SL01", "dengue")`, `("SL02", "leptospirosis")`. Bilangin ang mga kaso ng dengue bawat `barangay_id`, i-join ang mga bilang sa `brgy` **gamit ang `barangay_id`**, lagyan ng 0 ang barangay na walang kaso, at idagdag ang `attack_rate` = cases / population × 10,000. Dapat 24 na row pa rin ang resulta.*

In [ ]:
import json

import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from scipy import stats

brgy = gpd.read_file("data/synthetic/ph/santa_luz_barangays.geojson")
pop = pd.read_csv("data/synthetic/ph/santa_luz_barangays.csv")[["barangay_id", "households"]]
ll = pd.read_csv("data/synthetic/ph/santa_luz_linelist.csv")

dengue_counts = ll[ll["disease"] == "dengue"].groupby("barangay_id").size().rename("cases")
brgy = brgy.merge(pop, on="barangay_id", how="left")
brgy = brgy.merge(dengue_counts, left_on="barangay_id", right_index=True, how="left")
brgy["cases"] = brgy["cases"].fillna(0).astype(int)        # 0 ang barangay na walang kaso, hindi missing
brgy["attack_rate"] = brgy["cases"] / brgy["population"] * 10_000

print(f"{len(brgy)} barangay, {brgy['cases'].sum()} kaso ng dengue")
print(brgy.sort_values("attack_rate", ascending=False)
      [["barangay", "band", "population", "cases", "attack_rate"]].head(5).round(1).to_string(index=False))

**I-check natin.** (1) 24 na row pa rin ba pagkatapos ng mga join? (2) Umaabot ba sa lahat ng 388 kaso ng dengue sa line list ang suma ng mga bilang bawat barangay? (3) Isang rate, mano-mano: may 29 na kaso at 6,688 residente ang Talaba → 29 ÷ 6,688 × 10,000 = 43.4. (4) May barangay bang **zero** ang kaso, at nakaligtas ba ito sa join? Hindi kailanman gumagawa ang `groupby` ng row para sa barangay na walang kaso — ang `fillna(0)` sa itaas ang nagpapanatili rito.

In [ ]:
assert len(brgy) == 24, "may barangay na nawala o nadoble sa isang join"
assert brgy["cases"].sum() == (ll["disease"] == "dengue").sum() == 388
talaba = brgy.set_index("barangay").loc["Talaba"]
assert abs(29 / 6688 * 10_000 - talaba["attack_rate"]) < 1e-9
print("Talaba, mano-mano:", round(29 / 6688 * 10_000, 1))
print("Mga barangay na zero ang kaso ng dengue:", brgy.loc[brgy["cases"] == 0, "barangay"].tolist())
print("Mga barangay sa resulta ng groupby:", len(dengue_counts), "<- kulang ng isa sa 24")

## Hakbang 2 — Bilang laban sa rate

Dalawang mapa ng parehong mga kaso. Sa kaliwa, **bilang**: nagliliwanag ang malalaking barangay dahil lang mas maraming nakatira roon. Sa kanan, **rate**: ang tsansang naging kaso ang isang residente. Sinasagot ng rate ang *"saan ang panganib?"*; sinasagot ng bilang ang *"saan ang trabaho?"* Parehong kapaki-pakinabang — huwag lang silang pagpalitin.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, column, title in [(axes[0], "cases", "Kaso ng dengue (bilang)"),
                          (axes[1], "attack_rate", "Attack rate bawat 10,000")]:
    brgy.plot(column=column, cmap="YlOrRd", edgecolor="#1c1b29", linewidth=0.6, legend=True,
              legend_kwds={"shrink": 0.6}, ax=ax)
    ax.set_title(title)
    ax.set_axis_off()
top_count = brgy.loc[brgy["cases"].idxmax()]
top_rate = brgy.loc[brgy["attack_rate"].idxmax()]
for ax, row in [(axes[0], top_count), (axes[1], top_rate)]:
    ax.annotate(row["barangay"], xy=(row.geometry.centroid.x, row.geometry.centroid.y),
                ha="center", fontsize=9, fontweight="bold",
                bbox={"boxstyle": "round,pad=0.2", "facecolor": "white", "alpha": 0.85})
fig.suptitle("Santa Luz City (kathang-isip), dengue 20 Hul - 30 Set 2026", fontsize=12)
plt.tight_layout()
plt.show()
print(f"Pinakamaraming kaso: {top_count['barangay']} ({top_count['band']}, {top_count['cases']} kaso, "
      f"{top_count['attack_rate']:.1f} bawat 10,000)")
print(f"Pinakamataas na rate: {top_rate['barangay']} ({top_rate['band']}, {top_rate['cases']} kaso, "
      f"{top_rate['attack_rate']:.1f} bawat 10,000)")

## Hakbang 3 — Class break at kulay

Pinapangkat ng choropleth ang mga value sa ilang **class**, at binabago ng mga break ang mensahe:

* **equal interval** — pare-pareho ang lapad ng mga class (dito, mga 11 bawat 10,000 ang bawat isa); puwedeng maiwan sa iisang kulay ang karamihan ng mapa dahil sa iilang sobrang taas na barangay;
* **quantile** — pare-pareho ang bilang ng barangay sa bawat class (24 ÷ 4 = 6); nagagamit ang bawat kulay, pero puwedeng magkalapit na magkalapit ang magkatabing class.

Kulay: gumamit ng **sequential** na palette (mapusyaw = mababa, madilim = mataas) gaya ng `YlOrRd` o `viridis`; nababasa ang mga ito nang sunod-sunod at kaya nila ang color-blindness at ang grey-scale na pag-print. Walang natural na pagkakasunod ang **rainbow** na palette (`jet`), at hinihila ng matingkad na dilaw sa gitna nito ang mata papunta sa mga *katamtamang* value.

In [ ]:
rate = brgy["attack_rate"]
equal_edges = np.linspace(rate.min(), rate.max(), 5)          # 4 na class na pare-pareho ang lapad
quantile_edges = rate.quantile([0, 0.25, 0.5, 0.75, 1]).to_numpy()   # 4 na class na tig-6 na barangay



def classes(values, edges, digits=1):
    # Hatiin ang mga value sa mga class na may label na "mababa-mataas" (kasama ang pinakamababang gilid)
    labels = [f"{edges[i]:,.{digits}f}-{edges[i + 1]:,.{digits}f}" for i in range(len(edges) - 1)]
    return pd.cut(values, edges, include_lowest=True, labels=labels)


brgy["equal_class"] = classes(rate, equal_edges)
brgy["quantile_class"] = classes(rate, quantile_edges)
print("Equal interval:", brgy["equal_class"].value_counts(sort=False).to_dict())
print("Quantile:      ", brgy["quantile_class"].value_counts(sort=False).to_dict())

fig, axes = plt.subplots(1, 3, figsize=(15, 5.2))
panels = [("equal_class", "YlOrRd", "Equal interval, sequential"),
          ("quantile_class", "YlOrRd", "Quantile, sequential"),
          ("quantile_class", "jet", "Quantile, rainbow (iwasan)")]
for ax, (column, cmap, title) in zip(axes, panels):
    brgy.plot(column=column, categorical=True, cmap=cmap, edgecolor="#1c1b29", linewidth=0.5,
              legend=True, legend_kwds={"loc": "upper center", "bbox_to_anchor": (0.5, 0.0), "ncol": 2,
                                        "fontsize": 8, "title": "bawat 10,000", "frameon": False}, ax=ax)
    ax.set_title(title)
    ax.set_axis_off()
plt.tight_layout()
plt.show()

## Hakbang 4 — Palundag-lundag ang maliliit na numero

May 3,000–5,000 residente ang mga barangay sa kabundukan at iilang kaso lang, kaya malaki ang galaw ng rate nila kapag nadagdagan o nabawasan ng isang kaso. Ipinapakita ng **exact Poisson 95% confidence interval** ng bilang kung gaano kalaki: ang ibig sabihin ng malapad na interval ay "huwag i-ranggo ang barangay na ito batay lang sa kulay nito".

In [ ]:
def poisson_ci(count, alpha=0.05):
    # Exact (Garwood) na confidence interval para sa isang Poisson count
    low = stats.chi2.ppf(alpha / 2, 2 * count) / 2 if count > 0 else 0.0
    high = stats.chi2.ppf(1 - alpha / 2, 2 * (count + 1)) / 2
    return low, high


ci = brgy["cases"].apply(lambda c: pd.Series(poisson_ci(c), index=["low", "high"]))
brgy["rate_low"] = ci["low"] / brgy["population"] * 10_000
brgy["rate_high"] = ci["high"] / brgy["population"] * 10_000
brgy["one_more_case"] = (brgy["cases"] + 1) / brgy["population"] * 10_000

cols = ["barangay", "band", "population", "cases", "attack_rate", "rate_low", "rate_high", "one_more_case"]
picked = brgy.set_index("barangay").loc[["Talaba", "Ilog", "Maligaya", "Bukid"]].reset_index()
print(picked[cols].round(1).to_string(index=False))

## Hakbang 5 — Isang interactive na mapa

Para sa screen (ang dashboard ng [kabanata 09](../09_dashboard.md)), puwedeng mag-hover ang mambabasa sa interactive na mapa para sa eksaktong numero. Kailangan ng Plotly ang mga polygon bilang GeoJSON at ang key na nag-uugnay sa bawat row sa isang polygon: `featureidkey="properties.barangay_id"`. Hindi tayo naglalagay ng background tile: kathang-isip ang Santa Luz, at sadyang nasa gitna ng dagat ang mga coordinate nito.

In [ ]:
geojson = json.loads(brgy[["barangay_id", "geometry"]].to_json())
fig = px.choropleth(
    brgy.drop(columns="geometry"),
    geojson=geojson,
    locations="barangay_id",
    featureidkey="properties.barangay_id",
    color="attack_rate",
    color_continuous_scale="YlOrRd",
    hover_name="barangay",
    hover_data={"band": True, "cases": True, "population": ":,", "attack_rate": ":.1f", "barangay_id": False},
    labels={"attack_rate": "bawat 10,000", "cases": "Kaso", "population": "Populasyon", "band": "Bahagi"},
    title="Dengue attack rate bawat 10,000, Santa Luz City (kathang-isip)",
)
fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(margin={"l": 0, "r": 0, "t": 40, "b": 0}, height=420)
fig.show()

## Nagkamali si Tala: ang dalawang barangay na naglaho

Nagpadala si Jun ng mabilisang table ng bilang ng dengue ayon sa **pangalan** ng barangay, mano-manong tinype. Pinakiusapan ni Maya si Tala na i-join ito sa mapa, at nag-join si Tala **gamit ang pangalan**. Mukhang ayos ang mapa — pero ang tinype ni Jun ay `Santo Nino` (walang ñ) at `Tabing Ilog` (walang gitling). Sa isang inner join, basta na lang nawawala ang dalawang row na iyon; walang error, walang babala. Blangko sana sa mapa ng Mayor ang dalawang barangay sa tabing-ilog na may 41 kaso sa kabuuan.

Isang linya lang ang bantay: `assert len(merged) == 24`. Ang solusyon ay mag-join gamit ang **code** (`barangay_id`), hindi kailanman gamit ang tinype na pangalan.

In [ ]:
# Mano-manong tinype na table ni Jun (parehong bilang, dalawang pangalang iba ang baybay)
jun = brgy[["barangay", "cases"]].copy()
jun["barangay"] = jun["barangay"].replace({"Santo Niño": "Santo Nino", "Tabing-Ilog": "Tabing Ilog"})

merged = brgy[["barangay_id", "barangay", "geometry"]].merge(jun, on="barangay", how="inner")   # ang join ni Tala
print("Mga row pagkatapos ng join ni Tala:", len(merged))
try:
    assert len(merged) == 24, f"join kept {len(merged)} of 24 barangays"
except AssertionError as err:
    print("Nahuli:", err)

check = brgy[["barangay"]].merge(jun, on="barangay", how="outer", indicator=True)
print(check[check["_merge"] != "both"].to_string(index=False))

## Hakbang 6 — Mag-zoom out: ang 18 rehiyon ng Pilipinas

Sa isang briefing, nakakatulong na ipakita kung nasaan sa bansa ang isang probinsiya. Nasa `ph_regions.topo.json` ang totoong hangganan ng 18 rehiyon bilang **TopoJSON** (isang siksik na pinsan ng GeoJSON); direkta itong nababasa ng geopandas. Ini-join natin ang totoong regional statistics sa `ph_regions_stats.csv` gamit ang PSGC code na `adm1_psgc` (numero sa parehong file) at iminamapa ang **population density noong 2024**. Kinuwenta ang density (populasyon noong 2024 ÷ sukat ng lupa), at may source URL sa CSV ang parehong input. Mga 22 beses na mas siksik ang NCR kaysa sa sumunod na rehiyon (CALABARZON), kaya kung equal-width na class, iisa ang kulay ng lahat ng iba pang rehiyon; ipinapakita ng quantile class ang mga pagkakaiba.

Isa itong **context** map ng totoong bansa. Huwag kailanman iguhit dito ang kathang-isip na Santa Luz.

In [ ]:
regions = gpd.read_file("data/synthetic/ph/ph_regions.topo.json").set_crs("EPSG:4326")
ph_stats = pd.read_csv("data/synthetic/ph/ph_regions_stats.csv", dtype={"region_id": str, "psgc": str})
ph = regions[["adm1_psgc", "adm1_en", "geometry"]].merge(ph_stats, on="adm1_psgc", how="inner")
assert len(ph) == 18, f"join kept {len(ph)} of 18 regions"

density = ph["density_2024_per_km2"]
ph["density_class"] = classes(density, density.quantile([0, 0.25, 0.5, 0.75, 1]).to_numpy(), digits=0)
print(ph.sort_values("density_2024_per_km2", ascending=False)
      [["region_short", "population_2024", "land_area_km2", "density_2024_per_km2"]].head(4).to_string(index=False))
print("Source ng population_2024:", ph["population_2024_source_url"].iloc[0])

fig, ax = plt.subplots(figsize=(6.5, 8.5))
ph.plot(column="density_class", categorical=True, cmap="YlGnBu", edgecolor="#1c1b29", linewidth=0.4,
        legend=True, legend_kwds={"loc": "upper right", "title": "Tao bawat km², 2024", "fontsize": 8}, ax=ax)
ax.set_title("Population density ayon sa rehiyon, Pilipinas, 2024")
ax.set_axis_off()
fig.text(0.02, 0.02, "Boundaries: NAMRIA, PSA via OCHA COD-AB (CC BY-IGO); TopoJSON: faeldon/philippines-json-maps (MIT).\n"
         "Density = populasyon 2024 / sukat ng lupa, parehong mula sa en.wikipedia.org/wiki/Regions_of_the_Philippines.",
         fontsize=6.5)
plt.tight_layout()
plt.show()

## Hakbang 7 — Gawing ligtas i-publish

Ang **dot map** ng mga bahay na may kaso ay listahan ng mga address na iginuhit bilang tuldok: kayang ituro ng sinumang nakakakilala sa kalye ang isang bahay. Sa loob ng CESU, puwede itong gabay ng mga 4S team; sa pampublikong pahina, kinikilala nito ang mga kapitbahay. Ang ipinapakita ng pampublikong mapa ay **pinagsama-samang polygon** (mga barangay), at kahit ganoon, maliit na cell ang barangay na may 1–4 na kaso at puwede itong magturo sa isang tao. Isang karaniwang patakaran, at ang ginagamit ng aklat na ito: **itago ang bilang na 1–4** (ipakita bilang "<5") at itago rin ang rate nila, dahil ibinabalik ng rate × populasyon ang bilang.

In [ ]:
public = brgy[["barangay_id", "barangay", "population", "cases", "attack_rate", "geometry"]].copy()
small = public["cases"].between(1, 4)
public["cases_public"] = public["cases"].astype(str).where(~small, "<5")
public["rate_public"] = public["attack_rate"].round(1).where(~small)        # NaN = hindi ipinapakita
print(f"Naka-suppress: {small.sum()} sa 24 na barangay:", public.loc[small, "barangay"].tolist())

fig, ax = plt.subplots(figsize=(7, 4.8))
public.plot(column="rate_public", cmap="YlOrRd", edgecolor="#1c1b29", linewidth=0.5, legend=True,
            legend_kwds={"shrink": 0.6, "label": "bawat 10,000"}, ax=ax,
            missing_kwds={"color": "#eeeeee", "hatch": "///", "edgecolor": "#6e7882", "label": "1-4 kaso: hindi ipinapakita"})
ax.set_title("Pampublikong bersiyon: dengue bawat 10,000 (naka-suppress ang mga cell na may 1-4 kaso)")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Ikaw naman

1. **Leptospirosis.** Ulitin ang Hakbang 1 para sa `disease == "leptospirosis"` at imapa ang rate. Aling bahagi ng lungsod ang nagliliwanag ngayon? *(Pahiwatig: palitan ang isang string, at tandaan ang `fillna(0)`.)*
2. **Limang class.** Ulitin ang Hakbang 3 na may 5 quantile class. Mag-isa pa rin ba ang Talaba sa pinakamataas na class? *(Pahiwatig: `rate.quantile(np.linspace(0, 1, 6))`.)*
3. **Ibang regional variable.** Imapa ang `poverty_incidence_2023_pct` sa halip na density, at idagdag sa credit line ang source URL nito mula sa column na `poverty_incidence_2023_pct_source_url`.

In [ ]:
# 1. Leptospirosis (panimula)
lepto = ll[ll["disease"] == "leptospirosis"].groupby("barangay_id").size().rename("lepto_cases")
brgy_l = brgy[["barangay_id", "barangay", "band", "population", "geometry"]].merge(
    lepto, left_on="barangay_id", right_index=True, how="left")
brgy_l["lepto_cases"] = brgy_l["lepto_cases"].fillna(0).astype(int)
brgy_l["lepto_rate"] = brgy_l["lepto_cases"] / brgy_l["population"] * 10_000
assert len(brgy_l) == 24
by_band = brgy_l.groupby("band")[["lepto_cases", "population"]].sum()   # pagsamahin muna: hindi rate ng buong bahagi ang average ng mga rate
print((by_band["lepto_cases"] / by_band["population"] * 10_000).round(1))

## Ang ginawa mo

* Ini-join ang mga bilang sa mga polygon **gamit ang code**, pinanatili ang mga barangay na zero ang kaso, at in-assert ang 24 na row.
* Rate ang iminapa, hindi bilang, kapag panganib ang tanong; sadyang pumili ng class break at sequential na palette.
* Binasa ang malalapad na confidence interval bilang "huwag i-ranggo batay lang sa kulay".
* Iginuhit ang totoong context map ng 18 rehiyon kasama ang credit line nito.
* Naghanda ng pampublikong mapa na naka-suppress ang maliliit na cell.

Susunod: ilalagay ng [kabanata 09](../09_dashboard.md) ang curve, ang threshold at ang mapang ito sa iisang screen.